# ConnectX - Kaggle Environment

In [ ]:
"""from kaggle_environments import make
env = make("connectx")
print(env.name, env.version)
print("Default Agents: ", *env.agents)"""

## TLDR;

In [ ]:
"""def agent(observation, configuration):
    board = observation.board
    columns = configuration.columns
    return [c for c in range(columns) if board[c] == 0][0]

env = make("connectx", debug=True)
# play agent above vs default random agent.
env.run([agent, "negamax"])
env.render(mode="ipython", width=600, height=500, header=False)"""

## Specification

In [ ]:
"""import json
print("Configuration:", json.dumps(env.specification.configuration, indent=4, sort_keys=True))
print("Observation:", json.dumps(env.specification.observation, indent=4, sort_keys=True))
print("Action:", json.dumps(env.specification.action, indent=4, sort_keys=True))"""

## Training using Gym

In [ ]:
"""class ConnectX(gym.Env):

    def __init__(self):
        self.env = make("connectx", debug=True)
        self.trainer = self.env.train([None, "random"])
        
        # Define required gym fields (examples):
        config = self.env.configuration
        self.action_space = gym.spaces.Discrete(config.columns)
        self.observation_space = gym.spaces.Discrete(config.columns * config.rows)

    def step(self, action):
        return self.trainer.step(action)
    
    def reset(self):
        return self.trainer.reset()
    
    def render(self, **kwargs):
        return self.env.render(**kwargs)
        
    
env = ConnectX()

done = False
obs = env.reset()
while not done:
    # Choose first available empty column as the action.
    action = [i for i in range(len(obs.board)) if obs.board[i] == 0][0]
    obs, reward, done, info = env.step(action)
env.render()
"""


Entraînement de l'agent avec algotithme POO

In [ ]:
#Les imports necessaires
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3 import PPO

from kaggle_environments import make
import gymnasium as gym

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
#Préprocessing 

def preprocess(observation, configuration):
    # On transforme la liste plate en grille 2D (6, 7)
    grid = np.array(observation.board).reshape(configuration.rows, configuration.columns)
    
    # On crée les 3 couches (canaux)
    # Couche 1 : Tes pions (1 si c'est ton mark, 0 sinon)
    # Couche 2 : Les pions de l'adversaire (1 si c'est l'autre mark, 0 sinon)
    # Couche 3 : Les cases vides (1 si c'est 0, 0 sinon)
    
    mark = observation.mark
    opp_mark = 3 - mark
    
    layer_me = (grid == mark).astype(np.float32)
    layer_opp = (grid == opp_mark).astype(np.float32)
    layer_empty = (grid == 0).astype(np.float32)
    
    # On empile les couches pour créer un bloc de forme (3, 6, 7)
    tensor = np.stack([layer_me, layer_opp, layer_empty], axis=0)
    
    return tensor #tenseur (3, 6, 7)

In [ ]:
#Architecture CNN

class ConnectXNet(nn.Module):
    def __init__(self):
        super(ConnectXNet, self).__init__()
        
        # 1. ÉTAPE DE SCAN (Convolution)
        # On utilise 32 filtres de taille 4x4 pour repérer les alignements
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=4, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        
        # 2. ÉTAPE D'APLATISSEMENT (Flatten)
        # On transforme les grilles en une liste de chiffres pour la logique
        self.flatten = nn.Flatten()
        
        # 3. ÉTAPE DE RAISONNEMENT (Couches Linéaires)
        # On calcule le nombre de neurones après convolution : 64 filtres * 5 lignes * 6 colonnes
        self.fc1 = nn.Linear(64 * 5 * 6, 128) # 128 neurones pour réfléchir
        
        # 4. DÉCISION FINALE
        # 7 neurones de sortie (un pour chaque colonne du jeu)
        self.fc2 = nn.Linear(128, 7)

    def forward(self, x):
        # Passage dans les scanners avec activation ReLU
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        
        # Transformation en liste
        x = self.flatten(x)
        
        # Réflexion logique
        x = F.relu(self.fc1(x))
        
        # Scores finaux pour les 7 colonnes
        return self.fc2(x)

In [ ]:
#wrapper gym

class ConnectX(gym.Env):
    def __init__(self):
        # On initialise le jeu officiel ConnectX de Kaggle
        self.env = make("connectx", debug=True)
        
        # On crée un "entraîneur" : l'IA jouera toujours contre un agent qui joue au hasard
        # None signifie que c'est la place de TON IA (Joueur 1)
        self.trainer = self.env.train([None, "random"])
        
        # On récupère les dimensions du plateau (6 lignes, 7 colonnes par défaut)
        config = self.env.configuration
        
        # On définit ce que l'IA a le droit de "voir" (Espace d'Observation) :
        # Un bloc de 3 couches (Moi, Adversaire, Vide) de 6x7 cases
        self.observation_space = gym.spaces.Box(
            low=0, high=1, 
            shape=(3, config.rows, config.columns), 
            dtype=np.float32
        )
        
        # On définit ce que l'IA a le droit de "faire" (Espace d'Action) :
        # Elle doit choisir un chiffre entre 0 et 6 (les 7 colonnes)
        self.action_space = gym.spaces.Discrete(config.columns)

    def step(self, action):
        """
        Cette fonction est le 'moteur' : elle est appelée à chaque fois 
        que l'IA choisit une colonne où faire tomber son pion.
        """
        # 1. RÉCUPÉRATION DE L'ÉTAT ACTUEL
        # On regarde le plateau juste avant que l'IA ne joue
        board = self.env.state[0].observation.board
        
        # 2. VÉRIFICATION DE LA VALIDITÉ DU COUP
        # Si la case tout en haut de la colonne choisie n'est pas vide (!= 0),
        # cela veut dire que la colonne est déjà pleine !
        if board[int(action)] != 0:
            # SANCTION IMMÉDIATE : L'IA perd 10 points et la partie s'arrête net (True)
            # C'est ainsi qu'on lui apprend à respecter les règles du jeu
            return preprocess(self.env.state[0].observation, self.env.configuration), -10.0, True, False, {}

        # 3. EXÉCUTION DU COUP
        # Si le coup est valide, on demande au moteur de Kaggle de faire tomber le pion
        # 'obs' = nouveau plateau, 'reward' = résultat, 'done' = est-ce fini ?
        obs, reward, done, info = self.trainer.step(int(action))
        
        # Correction technique : si Kaggle renvoie 'None', on transforme en 0.0
        if reward is None: reward = 0.0
        
        # 4. CALCUL DE LA RÉCOMPENSE (Reward Shaping)
        # On ne donne pas juste 1 ou -1, on guide l'IA plus finement :
        
        if not done:
            # BONUS DE SURVIE : Elle gagne 0.01 point à chaque fois qu'elle joue un coup valide.
            # Cela l'incite à ne pas perdre tout de suite.
            reward = 0.01 
        
        if done and reward == 1:
            # VICTOIRE : Le Graal ! Elle reçoit 1.0 point.
            reward = 1.0
        elif done and reward == 0: 
            # DÉFAITE : Elle reçoit -1.0 point (Punition).
            reward = -1.0
            
        # On renvoie à l'IA le plateau transformé par 'preprocess' (3 couches)
        return preprocess(obs, self.env.configuration), float(reward), bool(done), False, info
    
    def reset(self, seed=None, options=None):
        """
        Appelée au début de chaque nouvelle partie. 
        On vide le plateau et on remet tout à zéro.
        """
        obs = self.trainer.reset()
        # On renvoie le plateau vide sous forme de 3 couches
        return preprocess(obs, self.env.configuration), {}
    
    def render(self, **kwargs):
        """Affiche visuellement le plateau de jeu si on le demande."""
        return self.env.render(**kwargs)

# On crée enfin l'objet 'env' qui contient toute cette logique
env = ConnectX()

print("L'environnement est prêt !")
# On vérifie que la forme (3, 6, 7) est bien prise en compte
print("Forme de l'observation :", env.observation_space.shape)

L'environnement est prêt !
Forme de l'observation : (3, 6, 7)


In [ ]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

# 1. L'ADAPTATEUR (Le connecteur entre ton CNN et le Coach)
# Stable Baselines a besoin d'un format spécifique. On crée donc une "boîte" 
# qui contient ton ConnectXNet pour qu'il s'emboîte parfaitement dans le Coach PPO.
class CustomCombinedExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=7):
        # On dit au Coach : "L'image arrive par ici, et à la fin, je te donne 7 scores" (un par colonne)
        super(CustomCombinedExtractor, self).__init__(observation_space, features_dim)
        
        # On place ton cerveau ConnectXNet à l'intérieur de cette boîte
        self.cnn = ConnectXNet()

    def forward(self, observations):
        # Quand l'IA reçoit une image du plateau, elle la passe directement à ton CNN
        # pour obtenir les scores de décision
        return self.cnn(observations)

# 2. LA CONFIGURATION (Le manuel d'instruction du Coach)
# On crée un dictionnaire (policy_kwargs) pour dire au Coach comment utiliser ta boîte personnalisée.
policy_kwargs = dict(
    features_extractor_class=CustomCombinedExtractor, # Utilise ma boîte personnalisée...
    features_extractor_kwargs=dict(features_dim=7),   # ...et attends-toi à recevoir 7 sorties.
)

# 3. L'INITIALISATION (Le choix du Coach et du Gymnase)
# On crée l'objet 'model' qui va gérer tout l'apprentissage
model = PPO(
    "CnnPolicy",           # On utilise une politique basée sur les images (CNN)
    env,                   # On l'envoie dans ton gymnase (le Wrapper ConnectX)
    policy_kwargs=policy_kwargs, # On lui donne tes réglages personnalisés
    verbose=1,             # Affiche les progrès (score, temps) pendant qu'il apprend
    learning_rate=0.0003   # La vitesse à laquelle il ajuste ses neurones (pas trop vite pour ne pas oublier)
)

print("L'entraînement commence...")
# On lance 10 000 parties d'entraînement. 
# L'IA va jouer, gagner, perdre, et s'améliorer grâce aux récompenses du Wrapper
model.learn(total_timesteps=10000)

# 4. LA SAUVEGARDE (Mise en mémoire du savoir)
# On enregistre tout ce que l'IA a appris dans un fichier
# Ce fichier contient les réglages optimaux de tes neurones.
model.save("mon_ia_connectx")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
L'entraînement commence...


KeyboardInterrupt: 

In [7]:
# On extrait les poids du CNN (le cerveau) depuis l'algorithme PPO
torch.save(model.policy.features_extractor.cnn.state_dict(), "poids_gagnants.pth")
print("Poids sauvegardés avec succès dans poids_gagnants.pth")

Poids sauvegardés avec succès dans poids_gagnants.pth


In [9]:
def agent(observation, configuration):
    # 1. Prépare le plateau (3 couches)
    state = preprocess(observation, configuration)
    state_tensor = torch.tensor(state).unsqueeze(0)
    
    # 2. Utilise le cerveau entraîné (PPO)
    # On accède au CNN via model.policy.features_extractor.cnn
    with torch.no_grad():
        output = model.policy.features_extractor.cnn(state_tensor)
    
    # 3. Récupère les scores et filtre les coups valides
    scores = output.numpy()[0]
    valid_moves = [c for c in range(configuration.columns) if observation.board[c] == 0]
    
    # 4. Choisit le meilleur coup
    best_move = max(valid_moves, key=lambda m: scores[m])
    
    return int(best_move)

In [18]:
from kaggle_environments import make

env = make("connectx", debug=True)
# play agent above vs default random agent.
env.run([agent, "negamax"]) 

print(env.render(mode="ansi")) # Affiche le plateau en texte brut

+---+---+---+---+---+---+---+
| 0 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 0 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 2 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+
| 2 | 0 | 1 | 0 | 0 | 0 | 0 |
+---+---+---+---+---+---+---+

